# Two pipelines on one slice

Every target ROI is measured two ways. Both use the **same ROI** (your finetuned shape, or the clicked template):

| | Pipeline A: pixels (`fus.histology`) | Pipeline B: cells (`fus.cells`) |
|---|---|---|
| unit | pixels in the ROI | NeuN cells whose centre is in the ROI |
| GFP+ means | pixel > 5 noise SD above local background | cell's mean anti-GFP, normalised over the section, > 0.0575 |
| reports | coverage = fraction of pixels GFP+ | fraction of cells GFP+, cell count, density |

The 0.0575 threshold is the log-Otsu split from the 3 × 3 mm pilot (`docs/gfp-cell-tagging.md`). Pipeline B exports each ROI at full resolution (~11 s), segments NeuN with StarDist in its own environment, then tags the cells: ~5 min for six targets.

Pick a section, run the cells, and check it before running `python -m fus.cells --mouse N --all`.

In [ ]:
%gui qt
from pathlib import Path

import matplotlib.pyplot as plt
import napari
import numpy as np
import pandas as pd
import tifffile
from napari.utils.colormaps import DirectLabelColormap

from fus import cells, histology, orientation, rois

ROOT = histology.REPO_ROOT
MOUSE = 2
SECTION = "slide04_s3"
RERUN = False      # True re-runs pipeline B for this section (~5 min)

summary_path = cells.RESULTS / f"mouse{MOUSE:02d}_cell_rois.csv"
done = summary_path.exists() and SECTION in set(pd.read_csv(summary_path).section)
crops_kept = all((cells.work_dir(MOUSE) / f"{SECTION}_T{t}.ome.tif").exists() for t in range(1, 7))
if RERUN or not done or not crops_kept:
    cells.measure_section(MOUSE, SECTION, keep_crops=True)   # keep crops for the viewer below

B = pd.read_csv(summary_path).query("section == @SECTION").set_index("target")
per_cell = pd.read_csv(cells.work_dir(MOUSE) / f"{SECTION}_cells.csv")
print(f"pipeline B: {len(per_cell):,} cells in {len(B)} ROIs, threshold {B.threshold.iloc[0]}")

In [ ]:
# Pipeline A for the same ROIs, and each target's dose.
slots = pd.read_csv(ROOT / f"results/histology/mouse{MOUSE:02d}_template_slots.csv")
a = slots[(slots.section == SECTION) & (slots.channel == "TRITC")].copy()
if "target" not in a or a.target.isna().any():   # template rows carry slots only
    sheet = orientation.read_sheet()
    m = orientation.KNOWN_MIRRORED.get((MOUSE, SECTION))
    m = orientation.is_mirrored(sheet[(MOUSE, SECTION)], orientation.plus_x_side(sheet)) if m is None else m
    a["target"] = [histology.slot_to_target(s, m) for s in a.slot]
A = a.set_index(a.target.astype(int))
dose = pd.read_csv(ROOT / "results/histology/slices/doses.csv").query("mouse == @MOUSE").set_index("target").dose

table = pd.DataFrame({
    "dose": dose.round(3),
    "A: pixel coverage": A.coverage.round(3),
    "B: fraction cells GFP+": B.fraction_gfp_pos,
    "B: GFP+ cells": B.n_gfp_pos,
    "B: cells": B.n_cells,
    "B: cells / mm²": B.cells_per_mm2.round(0),
    "B: median cell GFP": B.median_cell_gfp,
}).rename_axis("target")
display(table)

In [ ]:
BLUE, ORANGE, INK, MUTED = "#2a78d6", "#eb6834", "#0b0b0b", "#898781"
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.6))

ax1.plot([0, 1], [0, 1], color=MUTED, linestyle=":", linewidth=1)
ax1.scatter(table["A: pixel coverage"], table["B: fraction cells GFP+"], s=260, color=BLUE,
            edgecolors="white", linewidths=2, zorder=3)
for t, r in table.iterrows():
    ax1.annotate(str(t), (r["A: pixel coverage"], r["B: fraction cells GFP+"]), ha="center",
                 va="center_baseline", color="white", weight="bold", zorder=4)
ax1.set(xlim=(0, 1), ylim=(0, 1))
ax1.set_xlabel("A: GFP+ pixel coverage", fontsize=11)
ax1.set_ylabel("B: fraction of cells GFP+", fontsize=11)
ax1.set_title("Pipeline B vs A (dotted: equal)", loc="left", fontsize=12, weight="bold")

for col, colour, name in (("A: pixel coverage", ORANGE, "A: pixel coverage"),
                          ("B: fraction cells GFP+", BLUE, "B: fraction of cells GFP+")):
    ax2.scatter(table.dose, table[col], s=200, color=colour, edgecolors="white", linewidths=2,
                zorder=3, label=name)
    for t, r in table.iterrows():
        ax2.annotate(str(t), (r.dose, r[col]), ha="center", va="center_baseline", color="white",
                     weight="bold", fontsize=8.5, zorder=4)
ax2.set_ylim(0, 1)
ax2.set_xlabel("acoustic dose (cumulative 2nd harmonic)", fontsize=11)
ax2.set_ylabel("delivery", fontsize=11)
ax2.set_title("Both vs dose (number = target)", loc="left", fontsize=12, weight="bold")
ax2.legend(frameon=False, loc="upper left")
for ax in (ax1, ax2):
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# napari: whole section (overview) + each ROI at full resolution with cells outlined.
# Yellow = GFP+, magenta = GFP-; cells outside the ROI are not drawn. Front up, notch left.
ds4 = cells.ds4_path(MOUSE, SECTION)
sec = histology.load_section(ds4)
ds4_um = sec.pixel_um
full_um = ds4_um / cells.DS4
lo, hi = B.norm_lo.iloc[0], B.norm_hi.iloc[0]
shapes, _ = cells.section_rois(MOUSE, SECTION)

viewer = napari.Viewer(title=f"Pipelines A and B — Mouse {MOUSE} {SECTION}")
viewer.add_image(sec.channels["CY5"], name="overview NeuN", colormap="gray", opacity=0.4,
                 blending="additive", scale=(ds4_um, ds4_um), units="um",
                 contrast_limits=tuple(np.percentile(sec.channels["CY5"][::4, ::4], [1, 99.8])))
viewer.add_image(sec.channels["TRITC"], name="overview GFP stain", colormap="green",
                 blending="additive", scale=(ds4_um, ds4_um), units="um", contrast_limits=(lo, hi))

for t, roi in sorted(shapes.items()):
    x, y, _, _ = cells.box(roi, full_um)
    crop = tifffile.TiffFile(cells.work_dir(MOUSE) / f"{SECTION}_T{t}.ome.tif").series[0].levels[0].asarray()
    place = dict(scale=(full_um, full_um), translate=(y * full_um, x * full_um), units="um")
    viewer.add_image(crop[cells.NEUN], name=f"T{t} NeuN", colormap="gray", opacity=0.4,
                     blending="additive", contrast_limits=tuple(np.percentile(crop[cells.NEUN][::8, ::8], [1, 99.8])),
                     **place)
    viewer.add_image(crop[cells.GFP], name=f"T{t} GFP stain", colormap="green", blending="additive",
                     contrast_limits=(lo, hi), **place)
    labels = tifffile.imread(cells.work_dir(MOUSE) / f"{SECTION}_T{t}_labels.tif")
    c = per_cell[per_cell.target == t]
    colours = {None: "transparent", 0: "transparent"}
    colours.update({int(l): ("yellow" if p else "magenta") for l, p in zip(c.label, c.gfp_positive)})
    layer = viewer.add_labels(labels, name=f"T{t} cells ({int(c.gfp_positive.sum())}/{len(c)} GFP+)", **place)
    layer.colormap = DirectLabelColormap(color_dict=colours)
    layer.contour = 2

viewer.add_shapes([roi.corners() for _, roi in sorted(shapes.items())], shape_type="ellipse",
                  name="ROIs", edge_color="white", face_color="transparent", edge_width=6,
                  scale=(ds4_um, ds4_um), units="um")

# Straighten (front up, notch left) in µm, applied after each layer's own scale/translate.
a = orientation.read_sheet()[(MOUSE, SECTION)]
affine = orientation.canonical_affine(a, scale=1 / ds4_um)
if affine is not None:
    for layer in viewer.layers:
        layer.affine = affine
viewer.canvas.overlays.scale_bar.visible = True
viewer.reset_view()

Zoom into a target to check the calls. Yellow outlines should sit on GFP-filled cells; magenta should be dim. Cells outside the ROI circle are segmented but not drawn or counted.

When this looks right: `.venv/bin/python -m fus.cells --mouse 2 --all` (and `--mouse 1`) runs pipeline B on every clicked section, writing `results/histology/cells/mouseNN_cell_rois.csv`. Crops are deleted after measuring unless you add `--keep-crops`.